In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)
print(v1.shape)

In [ ]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)
print(dv.shape)

In [ ]:
v1.dot(dv)

In [ ]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [ ]:
v2.dot(dv)

In [ ]:
from ingest import load_faq_data

documents = load_faq_data()

In [ ]:
documents[10]

In [ ]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [ ]:
len(texts)

In [ ]:
from tqdm.auto import tqdm

In [ ]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

In [ ]:
vectors[10].shape

In [ ]:
scores = []

for i in range(len(vectors)):
    score = v1.dot(vectors[i])
    scores.append(score)

In [ ]:
scores

In [ ]:
import numpy as np
X = np.array(vectors)

In [ ]:
scores = X.dot(v1)
scores

In [ ]:
idx = np.argmax(scores)
idx, scores[idx]

In [ ]:
documents[2]

In [ ]:
top5 = np.argsort(-scores)[:5]
top5

In [ ]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

In [ ]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [ ]:
vindex.search(v1, num_results=5, filter_dict = {"course": "llm-zoomcamp"}  )

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [ ]:
from rag_helper_vector import RAGVector

vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

In [ ]:
query = "I just found out about the program, can I still sign up?"
vector_assistant.rag(query)

In [ ]:
vector_assistant.rag("the program has already begun, can I still sign up?")

In [ ]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [ ]:
vs_index.fit(vectors, documents)

In [ ]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [ ]:
results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [ ]:
results

In [ ]:
vs_index.close()